# Create unique ID and concatenate separate years into one dataframe

In [1]:
import os
import sys

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.append("../utils")

import config

pd.set_option("display.max_columns", None)


### Import unclipped training geometries

In [2]:
def process_training_geometries(start_year, end_year):
    """
    Process training geometries for a range of years, standardize columns, and combine them into a single GeoDataFrame.
    Preserves the existing inspection_id values and sorts the final result by inspection_id.

    Args:
        start_year (int): The starting year of the range.
        end_year (int): The ending year of the range.

    Returns:
        GeoDataFrame: Combined, processed, and sorted GeoDataFrame.
    """

    def import_training_geometries(year):
        """
        Function to read in buffer geometry data for a given year and convert to Albers CRS.
        """
        training_geometries = os.path.join(
            config.data_dir,
            "training_geometries",
            f"training_geometries_{year}.geojson",
        )
        return gpd.read_file(training_geometries).to_crs(config.albers_crs)

    # Import and process geometries for each year
    inspections_dict = {
        year: import_training_geometries(year)
        for year in range(start_year, end_year + 1)
    }

    # Add a `year` column to each dataframe
    for year, df in inspections_dict.items():
        df["year"] = year

    # Combine all years into one dataframe
    training_inspections = pd.concat(
        inspections_dict.values(), axis=0, ignore_index=True
    )
    
    # Sort by inspection_id to ensure consistent ordering
    training_inspections = training_inspections.sort_values('inspection_id').reset_index(drop=True)

    return training_inspections

In [3]:
training_inspections = process_training_geometries(2019, 2023)
training_inspections

,inspection_id,apn,Date,year,month,status,geometry
0,1,None,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37..."
1,2,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37..."
2,3,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37..."
3,4,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37..."
4,5,None,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37..."
...,...,...,...,...,...,...,...
67575,67576,None,2023-10-31,2023,10,Compliant,"POLYGON ((-35016.13 -349482.83, -34938.314 -34..."
67576,67577,None,2023-11-15,2023,11,Compliant,"POLYGON ((-36511.881 -350331.419, -36432.622 -..."
67577,67578,None,2023-11-15,2023,11,Compliant,"POLYGON ((-38782.099 -346106.741, -38702.857 -..."
67578,67579,None,2023-11-15,2023,11,Compliant,"POLYGON ((-38988.592 -346166.568, -38884.895 -..."


In [4]:
# Write to file
training_inspections.to_file(
    os.path.join(
        config.data_dir,
        "PUZZLE_PIECES",
        "inspections_master_training_geometries.geojson",
    ),
    driver="GeoJSON",
)

### Import clipped geometries

In [5]:
def process_buffer_geometries(start_year, end_year):
    """
    Process buffer geometries for a range of years, standardize columns, and combine them into a single GeoDataFrame.
    Preserves the existing inspection_id values and sorts the final result by inspection_id.

    Args:
        start_year (int): The starting year of the range.
        end_year (int): The ending year of the range.

    Returns:
        GeoDataFrame: Combined, processed, and sorted GeoDataFrame.
    """

    def import_buffer_geometries(year):
        """
        Function to read in buffer geometry data for a given year and convert to Albers CRS.
        """
        buffer_geometries = os.path.join(
            config.data_dir,
            "buffer_geometries",
            f"buffer_geometries_{year}.geojson",
        )
        return gpd.read_file(buffer_geometries).to_crs(config.albers_crs)

    # Import and process geometries for each year
    inspections_dict = {
        year: import_buffer_geometries(year) for year in range(start_year, end_year + 1)
    }

    # Add a `year` column to each dataframe
    for year, df in inspections_dict.items():
        df["year"] = year
        # Reset index to ensure unique indices before concatenation
        inspections_dict[year] = df.reset_index(drop=True)

    # Find common columns across all dataframes
    common_cols = set.intersection(
        *[set(df.columns) for df in inspections_dict.values()]
    )

    # Only keep common columns before concatenation
    for year in inspections_dict:
        inspections_dict[year] = inspections_dict[year][list(common_cols)]

    # Combine all years into one dataframe
    buffer_inspections = pd.concat(inspections_dict.values(), axis=0, ignore_index=True)
    
    # Sort by inspection_id to ensure consistent ordering
    buffer_inspections = buffer_inspections.sort_values('inspection_id').reset_index(drop=True)

    return buffer_inspections

In [6]:
buffer_inspections = process_buffer_geometries(2019, 2023)
buffer_inspections

/tmp/ipykernel_244816/4052484543.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  buffer_inspections = pd.concat(inspections_dict.values(), axis=0, ignore_index=True)
/tmp/ipykernel_244816/4052484543.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  buffer_inspections = pd.concat(inspections_dict.values(), axis=0, ignore_index=True)


,address_ad,deckporche,occupantho,water_sour,propanetan,b_removeleavesneedlesveg,can_engine_access_water_source,reinspectiondate,numberfield1,e_removefl,address_sub_thoroughfare,propanetankdistance,created_at,roofconstr,water_comm,i_removefuelsusingctguidelines,numberfiel,waterstora,exteriorsiding,keyid,can_engine,previousyrinspectionstatus,system_upd,l_logsstum,apn,address_lo,structuret,previousyr,address_suite,a_removebranchesfromstovepipe,address_fu,g_relocateexposedwoodpiles,windowpane,system_cre,numberfield2,citationnu,status,creator,m_outbuild,propertyst,address_su,c_removede,fenceattachedtostructure,assigned_t,n_displaya,inspectorfirstname,utilitym_1,editeddate,address_po,fulcrum_id,h_cutannua,water_source,calculated,shift_julian,o_stovepip,i_removefu,globalid,year,escalatetocoordinator,utilitymiscstucturecount,inspectionstatus,inspection,le100number,water_storage_size_stored_water_on_individual_parcels_only,water_comments,latitude,address__1,longitude,numberofst,previousyrinspecteddata,appenddate,previous_1,gatecode,calculateduid,firescopeid,number_of_,f_removefl,Date,deckporchg,utilitymiscstructuredistance,calfireuni,n_displayaddresscontrasting,yearbuilt,j_exposedw,geometry,prevention_inspectorlastname,numberofstructures,address_co,address_admin_area,shift_juli,system_updated_at,n_addressd,deckporchelevated,report_tit,address__2,recommendclearvegetation,coredata,escalateto,textfield1,a_removebr,k_removedeaddyingwoodyfuels,inspection_id,number_of_dead_trees_within_300_ft_of_residence,recommend_,editor,h_cutannualgrassesforbs,d_removede,address_postal_code,createdby,prevention_inspectorlastname_other,j_exposedwoodpiles,photoid_caption,b_removele,stationn_1,inspectorp,editdate,inspectionhours,deckporchgrade,accessegre,county,citationda,community,address_locality,addressvisible,address_country,calculatededitor,month,c_removedeaddyingtrees,inspecti_3,structurehabitable,f_removeflammablevegetation,k_removede,version,calculatedglobalid,accessegress,inspectiondate_calculate,patiocover,o_chimneys,citationnumber,exteriorsi,structureh,creationda,principal_,occupanthome,enginenumber,updated_at,inspectorposition,photoid,address_full,createddat,updated_by,fenceattac,clearreins,addressvis,propertystatus,stationnam,community_,numberfi_1,e_removedy,nonhabitableoutbuildings,g_relocate,calculat_1,system_created_at,editedby,m_outbuildingsliquidpropanegas,structuretype,comments,project,address_sub_admin_area,deliveryno,ventscreen,o_stovepipemetalscreenopenings,inspectioncount,responsibi,j_allexpos,siteaddres,partner_di,address_thoroughfare,le100numbe,photoid_url,battalion,core_with_,firescopei,calculateddate,enginenumb,i_reducefu,eaves,created_by,nonhabitab,creationdate,prevention_inspectorfirstname_other,roofconstruction,calculatededitdate,deadtrees3,coredatafl,patiocovercarport,reinspecti,community_other,report_title,shift,calculat_2,utilitymis,inspecti_2,l_removelogstumpsembeddedsoil,assigned_to,k_deaddyin,photoid_ca,l_removelo,inspecti_1,recommendc,d_removedeaddyinggrassplants,calculat_4,editstatus,water_stor,inspectorlastname,textfield2,calculat_3,address_th,prevention_inspectorfirstname,calfireunit,stationname,e_removeflammablegroundcover,siteaddress,deliverynotificationmethod
0,CA,No Deck/Porch,Yes,Private Hydrant- Private Stored Water,>30',None,None,NaT,None,None,None,None,2018-05-25 16:25:06,Metal,None,None,0.0,yes,None,NaN,yes,None,2019-06-05 16:35:26,None,141-010-040,Santa Ynez,Utility or Miscellaneous Structure > 120 sqft,None,None,None,3462 Brinkerhoff Rd Santa Ynez Santa Barbara C...,None,No Windows,2019-05-07 11:52:36,None,None,Compliant,None,None,None,3462,None,None,None,None,None,0,None,93460,d9a4d031-d16d-4c84-b32c-f3fcd7d945df,None,None,None,None,None,None,None,2019,None,None,None,2019-06-05,None,None,None,34.669616,None,-120.031160,0.0,None,2019-06-05,None,None,None,None,NaN,None,2019-06-05,No Deck/Porch,None,SBC,None,2000.0,None,"POLYGON ((-2813.179 -371821.557, -2813.179 -37...",None,None,US,None,None,NaT,None,None,D

In [7]:
# Write to file
buffer_inspections.to_file(
    os.path.join(config.data_dir, "PUZZLE_PIECES", "inspections_master.geojson"),
    driver="GeoJSON",
)

## Check that the inpection ids match for training geometry and buffer geometries

In [8]:
# Create dataframe of just compliance status and inspection_id
status_training = training_inspections.reindex(columns=["inspection_id", "status"])
status_training["status"] = status_training["status"].map(
    {"Compliant": 0, "Non-Compliant": 1}
)

In [9]:
status_buffer = buffer_inspections.reindex(columns=["inspection_id", "status"])
status_buffer["status"] = status_buffer["status"].map(
    {"Compliant": 0, "Non-Compliant": 1}
)

In [10]:
status_training_sorted = status_training.sort_values('inspection_id').reset_index(drop=True)
status_buffer_sorted = status_buffer.sort_values('inspection_id').reset_index(drop=True)
are_equal = status_training_sorted.equals(status_buffer_sorted)
are_equal

True